# ??? ?? ????? ???? ?????? ??? ? ?????
# ?? 3??? ???? ???????? ^^;;;

# ??????

In [ ]:
import pandas
import tensorflow
import matplotlib
from konlpy.tag import Mecab
from keras.preprocessing.text import Tokenizer
from keras.preprocessing.sequence import pad_sequences
import re


def preprocess_sentences(kor_sentences, eng_sentences, num_words=10000):
    # ??? ??? ???
    mecab = Mecab()

    # ?? ??? ??? ?? set? ??
    # kor_sentences = list(set([re.sub(r'[^?-??-??-?. ]', "", sentence) for sentence in kor_sentences]))
    # eng_sentences = list(set([re.sub(r'[^a-zA-Z. ]', "", sentence).lower() for sentence in eng_sentences]))
    
    # ???? ????, ???? ?? ??? ?? ?????.
    cleaned_corpus = []
    for kor, eng in zip(kor_sentences, eng_sentences):
        kor = re.sub(r'[^?-??-??-?. ]', "", kor)
        eng = re.sub(r'[^a-zA-Z. ]', "", eng).lower()

        # ?? ? ? ?? ? ??? ?? ??? ?? ?? ?? ?????.
        if kor and eng:
            cleaned_corpus.append((kor, eng))

    # ???
    kor_sentences, eng_sentences = zip(*cleaned_corpus)
    kor_sentences = [mecab.morphs(sentence) for sentence in kor_sentences]
    eng_sentences = [sentence.split() for sentence in eng_sentences]

    # ?? ???
    kor_tokenizer = Tokenizer(num_words=num_words)
    kor_tokenizer.fit_on_texts(kor_sentences)
    kor_sequences = kor_tokenizer.texts_to_sequences(kor_sentences)

    eng_tokenizer = Tokenizer(num_words=num_words)
    eng_tokenizer.fit_on_texts(eng_sentences)
    eng_sequences = eng_tokenizer.texts_to_sequences(eng_sentences)

    # ?? ??? ?? ?? ??
    kor_sequences = [[1] + seq + [2] for seq in kor_sequences]
    eng_sequences = [[1] + seq + [2] for seq in eng_sequences]

    # ??
    kor_sequences = pad_sequences(kor_sequences, padding='post')
    eng_sequences = pad_sequences(eng_sequences, padding='post')

    return kor_sequences, eng_sequences, kor_tokenizer, eng_tokenizer, cleaned_corpus


# ???? ??

In [ ]:
from sklearn.model_selection import train_test_split


?????? = "/aiffel/aiffel/korean-english-park.train.ko" #??? ??
?????? = "/aiffel/aiffel/korean-english-park.train.en" #??? ??

# ??? ?? ??
with open(??????, "r") as f:
    kor_sentences = f.readlines()

with open(??????, "r") as f:
    eng_sentences = f.readlines()

# ??? ??
kor_sequences, eng_sequences, kor_tokenizer, eng_tokenizer, cleaned_corpus = preprocess_sentences(kor_sentences, eng_sentences)

# ?? ???? ?? ???? ??
kor_train, kor_val, eng_train, eng_val = train_test_split(kor_sequences, eng_sequences, test_size=0.1)

# ??? ?? ??
for i in range(5):
    print("Korean : ", ' '.join([kor_tokenizer.index_word[idx] for idx in kor_sequences[i] if idx != 0]))
    print("English : ", ' '.join([eng_tokenizer.index_word[idx] for idx in eng_sequences[i] if idx != 0]))
    print("\n")

In [ ]:
import tensorflow as tf
from tensorflow.keras.layers import Input, LSTM, Dense, Embedding, Masking
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam


BUFFER_SIZE = len(kor_sequences)
BATCH_SIZE = 32
steps_per_epoch = len(kor_sequences) // BATCH_SIZE
embedding_dim = 256
units = 512
kor_vocab_size = 10000
eng_vocab_size = 10000

# ?? ? ?? ???? ??
train_dataset = tf.data.Dataset.from_tensor_slices((kor_train, eng_train)).shuffle(BUFFER_SIZE)
train_dataset = train_dataset.batch(BATCH_SIZE, drop_remainder=True)

val_dataset = tf.data.Dataset.from_tensor_slices((kor_val, eng_val)).shuffle(BUFFER_SIZE)
val_dataset = val_dataset.batch(BATCH_SIZE, drop_remainder=True)

# ????

In [ ]:
import tensorflow as tf
from tensorflow.keras.layers import Input, LSTM, Dense, Embedding, Masking
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam

# Attention Layer
class BahdanauAttention(tf.keras.layers.Layer):
    def __init__(self, units):
        super(BahdanauAttention, self).__init__()
        self.W1 = Dense(units)
        self.W2 = Dense(units)
        self.V = Dense(1)

    def call(self, query, values):
        query_with_time_axis = tf.expand_dims(query, 1)
        score = self.V(tf.nn.tanh(self.W1(query_with_time_axis) + self.W2(values)))
        attention_weights = tf.nn.softmax(score, axis=1)
        context_vector = attention_weights * values
        context_vector = tf.reduce_sum(context_vector, axis=1)
        return context_vector, attention_weights


class Encoder(tf.keras.Model):
    def __init__(self, vocab_size, embedding_dim, enc_units):
        super(Encoder, self).__init__()
        self.enc_units = enc_units
        self.embedding = Embedding(vocab_size, embedding_dim)
        self.gru = tf.keras.layers.GRU(self.enc_units,
                                       return_sequences=True,
                                       return_state=True)

    def call(self, x, hidden):
        x = self.embedding(x)
        output, state = self.gru(x, initial_state=hidden)
        return output, state

    def initialize_hidden_state(self, batch_sz):
        return tf.zeros((batch_sz, self.enc_units))


class Decoder(tf.keras.Model):
    def __init__(self, vocab_size, embedding_dim, dec_units):
        super(Decoder, self).__init__()
        self.dec_units = dec_units
        self.embedding = Embedding(vocab_size, embedding_dim)
        self.gru = tf.keras.layers.GRU(self.dec_units,
                                       return_sequences=True,
                                       return_state=True)
        self.fc = Dense(vocab_size)
        self.attention = BahdanauAttention(self.dec_units)

    def call(self, x, hidden, enc_output):
        context_vector, attention_weights = self.attention(hidden, enc_output)
        x = self.embedding(x)
        x = tf.concat([tf.expand_dims(context_vector, 1), x], axis=-1)
        output, state = self.gru(x)
        output = tf.reshape(output, (-1, output.shape[2]))
        x = self.fc(output)
        return x, state, attention_weights


# Parameters
BUFFER_SIZE = len(kor_sequences)
BATCH_SIZE = 32
steps_per_epoch = len(kor_sequences) // BATCH_SIZE
embedding_dim = 256
units = 512
kor_vocab_size = 10000
eng_vocab_size = 10000

# Encoder, Decoder ??
encoder = Encoder(kor_vocab_size, embedding_dim, units)
decoder = Decoder(eng_vocab_size, embedding_dim, units)

# Optimizer, Loss ??
optimizer = Adam()
loss_object = tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True, reduction='none')

def loss_function(real, pred):
    mask = tf.math.logical_not(tf.math.equal(real, 0))
    loss_ = loss_object(real, pred)
    mask = tf.cast(mask, dtype=loss_.dtype)
    loss_ *= mask
    return tf.reduce_mean(loss_)

# Training
@tf.function
def train_step(inp, targ, enc_hidden):
    loss = 0

    with tf.GradientTape() as tape:
        enc_output, enc_hidden = encoder(inp, enc_hidden)
        dec_hidden = enc_hidden
        dec_input = tf.expand_dims([1] * BATCH_SIZE, 1)  # '<start>' is replaced with 1

        # Teacher forcing - feeding the target as the next input
        for t in range(1, targ.shape[1]):
            # passing enc_output to the decoder
            predictions, dec_hidden, _ = decoder(dec_input, dec_hidden, enc_output)
            loss += loss_function(targ[:, t], predictions)

            # using teacher forcing
            dec_input = tf.expand_dims(targ[:, t], 1)

    batch_loss = (loss / int(targ.shape[1]))
    variables = encoder.trainable_variables + decoder.trainable_variables
    gradients = tape.gradient(loss, variables)
    optimizer.apply_gradients(zip(gradients, variables))
    
    return batch_loss





# ??

In [ ]:
# ??
EPOCHS = 3

for epoch in range(EPOCHS):
    print("#2")
    enc_hidden = encoder.initialize_hidden_state(BATCH_SIZE)
    total_loss = 0
    for (batch, (inp, targ)) in enumerate(train_dataset.take(steps_per_epoch)):
        batch_loss = train_step(inp, targ, enc_hidden)
        total_loss += batch_loss
    print(f'Epoch {epoch+1} Loss {total_loss/steps_per_epoch:.4f}')

    # ??
    val_loss = 0
    val_steps_per_epoch = len(kor_val) // BATCH_SIZE
    for (batch, (inp, targ)) in enumerate(val_dataset.take(val_steps_per_epoch)):
        enc_hidden_val = encoder.initialize_hidden_state(BATCH_SIZE)
        enc_output_val, enc_hidden_val = encoder(inp, enc_hidden_val)
        dec_hidden_val = enc_hidden_val
        dec_input_val = tf.expand_dims([1] * BATCH_SIZE, 1)

        for t in range(1, targ.shape[1]):
            predictions, dec_hidden_val, _ = decoder(dec_input_val, dec_hidden_val, enc_output_val)
            val_loss += loss_function(targ[:, t], predictions)
            dec_input_val = tf.expand_dims(targ[:, t], 1)
    val_loss /= int(targ.shape[1])
    print(f'Validation Loss {val_loss/val_steps_per_epoch:.4f}')


# ????

In [ ]:
def preprocess_and_encode(sentence, mecab, tokenizer, max_len):
    # Preprocess and tokenize
    sentence = re.sub(r'[^?-??-??-?. ]', "", sentence)
    sentence = mecab.morphs(sentence)
    sentence = tokenizer.texts_to_sequences([sentence])

    # Add start and end token and pad sequence
    sentence = [[1] + seq + [2] for seq in sentence]
    sentence = pad_sequences(sentence, maxlen=max_len, padding='post')
    return sentence

def translate(sentence, encoder, decoder, mecab, kor_tokenizer, eng_tokenizer):
    # Preprocessing and encoding
    sentence = preprocess_and_encode(sentence, mecab, kor_tokenizer, kor_sequences.shape[-1])

    inputs = tf.convert_to_tensor(sentence)
    result = ''

    hidden = [tf.zeros((1, units))]
    enc_out, enc_hidden = encoder(inputs, hidden)

    dec_hidden = enc_hidden
    dec_input = tf.expand_dims([1], 0)

    for t in range(eng_sequences.shape[-1]):
        predictions, dec_hidden, attention_weights = decoder(dec_input,
                                                             dec_hidden,
                                                             enc_out)

        predicted_id = tf.argmax(predictions[0]).numpy()

        # Skip for start and end tokens
        if predicted_id == 1 or predicted_id == 2:
            continue

        result += eng_tokenizer.index_word[predicted_id] + ' '

        dec_input = tf.expand_dims([predicted_id], 0)

    return result.strip(), sentence


# ????

In [ ]:
# Testing the function
sentences = ["???? ?????.", "???? ?? ?? ??.", "??? ?? ??.", "?? ?? ???? ????."]

for sentence in sentences:
    print(f'Input: {sentence}')
    result, _ = translate(sentence, encoder, decoder, Mecab(), kor_tokenizer, eng_tokenizer)
    print(f'Translated: {result}\n')
